## Load NAB time series

## Step 1 - Load NYC taxi dataset and attach anomaly labels
In this step, the raw `nyc_taxi.csv` file from the NAB `realKnownCause` folder is loaded into a
dataframe. The corresponding anomaly timestamps for this dataset are retrieved from
`combined_labels.json` using the key `realKnownCause/nyc_taxi.csv` and converted into a binary
`is_anomaly` column (0 = normal, 1 = labelled anomaly). The `timestamp` column is parsed as a
proper datetime type and the rows are sorted chronologically so that the NYC taxi demand series is
cleanly aligned in time and each anomaly label matches the correct 30-minute interval.


### A) Load raw nyc_taxi.csv dataset

In [14]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load one example file
file_path = os.path.join(base_path, "nyc_taxi.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2014-07-01 00:00:00,10844
1,2014-07-01 00:30:00,8127
2,2014-07-01 01:00:00,6210
3,2014-07-01 01:30:00,4656
4,2014-07-01 02:00:00,3820


### B) Load Anomaly Labels (JSON)

In [16]:
import json  # to work with JSON files

# Tell Python where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# Shows the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]



['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

### C) Extract Labels for Selected Dataset

In [17]:
# Pick out the labels for the NYC taxi dataset
dataset_key = "realKnownCause/nyc_taxi.csv"

nyc_taxi_labels = labels[dataset_key]

# Turn the list of timestamp strings into a DataFrame
nyc_taxi_labels_df = pd.DataFrame(
    {"timestamp": pd.to_datetime(nyc_taxi_labels)}
)

nyc_taxi_labels_df.head()

,timestamp
0,2014-11-01 19:00:00
1,2014-11-27 15:30:00
2,2014-12-25 15:00:00
3,2015-01-01 01:00:00
4,2015-01-27 00:00:00


### D) Add Labels to DataFrame

In [18]:
# Ensure main dataframe timestamps are in datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Convert the anomaly label timestamps to datetime
# use the list nyc_taxi_labels directly
anomaly_times = pd.to_datetime(nyc_taxi_labels)

# Create is_anomaly: 1 if this timestamp is labelled, else 0
df["is_anomaly"] = df["timestamp"].isin(anomaly_times).astype(int)

# Quick sanity check
df[["timestamp", "value", "is_anomaly"]].head()


,timestamp,value,is_anomaly
0,2014-07-01 00:00:00,10844,0
1,2014-07-01 00:30:00,8127,0
2,2014-07-01 01:00:00,6210,0
3,2014-07-01 01:30:00,4656,0
4,2014-07-01 02:00:00,3820,0


### E) Class Balance

In [19]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,10315,99.952
1,Anomaly,5,0.048


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [20]:
# Ensure timestamp is datetime and sorted
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Rename to common research project schema
df = df.rename(columns={"timestamp": "time"})

# Add case_study identifier
df["case_study"] = "nyc_taxi"

# Quick check
df = df[["time", "value", "is_anomaly", "case_study"]]

df.head()


,time,value,is_anomaly,case_study
0,2014-07-01 00:00:00,10844,0,nyc_taxi
1,2014-07-01 00:30:00,8127,0,nyc_taxi
2,2014-07-01 01:00:00,6210,0,nyc_taxi
3,2014-07-01 01:30:00,4656,0,nyc_taxi
4,2014-07-01 02:00:00,3820,0,nyc_taxi


In this step, the NYC taxi dataframe is aligned with the common schema used across all case
studies. The `timestamp` column is converted to a proper datetime type, renamed to `time`, and the
rows are sorted chronologically. A `case_study` column with the value `"nyc_taxi"` is added so that
this dataset can later be stacked or filtered alongside other case studies using the same column
names (`time`, `value`, `is_anomaly`, `case_study`).


## Step 3 - Missing values and Duplicates rows sanity check

### A) Missing values per column

In [25]:
# Missing values per column
missing_counts = df.isna().sum()

# Percentage of missing values in each column
missing_percentages = (df.isna().mean() * 100).round(3)

# Combine into a summary table
missing_summary = (
    pd.DataFrame(
        {
            "Column": missing_counts.index,
            "Missing count": missing_counts.values,
            "Missing (%)": missing_percentages.values,
        }
    )
    .sort_values("Missing count", ascending=False)
    .reset_index(drop=True)
)

missing_summary


,Column,Missing count,Missing (%)
0,time,0,0.0
1,value,0,0.0
2,is_anomaly,0,0.0
3,case_study,0,0.0


### B) Duplicate rows summary

In [24]:
# B) Duplicate rows summary
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_summary = pd.DataFrame(
    [
        ("Total rows", total_rows),
        ("Number of duplicate rows", duplicate_count),
        ("Duplicate rows (%)", round(duplicate_count / total_rows * 100, 3)),
    ],
    columns=["Statistic", "Value"],
)

duplicate_summary


,Statistic,Value
0,Total rows,10320.0
1,Number of duplicate rows,0.0
2,Duplicate rows (%),0.0


The missing-values table shows that the NYC taxi dataset has no missing values in any of the core
columns (`time`, `value`, `is_anomaly`, `case_study`). The duplicate-rows summary likewise reports
zero duplicate records. This confirms that the NAB NYC taxi series is clean at the basic data
quality level, so no additional imputation or de-duplication is needed. Documenting these negative
findings keeps the preprocessing steps transparent and easy to justify in the methodology chapter.
